# ASD-SCZ Bias Similarity: Gene Property Dissection

**Question:** What gene-level properties drive the shared cell-type bias pattern between ASD and SCZ?

**Approach:** Progressively remove genes ranked by constraint (LOEUF) or expression (BrainSpan),
then measure how the ASD-SCZ neuronal bias correlation changes compared to random removal.

**Key results:**
- Removing the most constrained genes (low LOEUF) rapidly destroys the ASD-SCZ correlation
- Removing by BrainSpan expression has a similar but weaker effect
- Constrained genes (LOEUF < 0.2) alone recapitulate the shared pattern; unconstrained genes do not

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from scipy import stats
from multiprocessing import Pool

import yaml
with open("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/config/config.yaml") as f:
    _cfg = yaml.safe_load(f)
PROJ_DIR = Path(_cfg["ProjDIR"])
sys.path.insert(0, str(PROJ_DIR / "src"))
from CellType_PSY import *

config = _cfg

HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

# Font setup
font_path = '/usr/share/fonts/truetype/msttcorefonts/Arial.ttf'
fm.fontManager.addfont(font_path)
fm._load_fontmanager(try_read_cache=False)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
# Gene weights
ASD_GW = Fil2Dict(str(PROJ_DIR / "dat/GeneWeights/HIQ.top61.nopLI.LGD_Dmis_SameWeight.bgmr.gw"))
SCZ_GW = Fil2Dict(str(PROJ_DIR / "dat/GeneWeights/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw"))

# Expression matrix
expression_matrix = config['analysis_types']['Centering']
HCT_Z2_MAT = pd.read_csv(str(PROJ_DIR / expression_matrix), index_col=0)
HCT_Z2_MAT.columns = HCT_Z2_MAT.columns.astype(int)
print(f"Expression matrix: {HCT_Z2_MAT.shape[0]} genes x {HCT_Z2_MAT.shape[1]} cell types")

In [ ]:
# Baseline ASD-SCZ correlation (all genes)
ASD_Bias = HumanCT_AvgZ_Weighted(HCT_Z2_MAT, ASD_GW)
ASD_Bias = AnnotateCTDat(ASD_Bias, Anno)

SCZ_Bias = HumanCT_AvgZ_Weighted(HCT_Z2_MAT, SCZ_GW)
SCZ_Bias = AnnotateCTDat(SCZ_Bias, Anno)

r_baseline, p_baseline = GetSingeCellBiasCorr(ASD_Bias, SCZ_Bias, efflabel="EFFECT", CTs=Neur_idx)
print(f"Baseline ASD-SCZ neuronal bias correlation: r={r_baseline:.3f}, p={p_baseline:.2e}")

## 2. Annotate Genes with Constraint and Expression

In [ ]:
# Load gnomAD v4 (primary) and v2 (fallback)
gnomad4 = pd.read_csv("/home/jw3514/Work/data/gnomad/gnomad.v4.0.constraint_metrics.tsv", sep="\t")
gnomad4 = gnomad4[(gnomad4["transcript"].str.contains('ENST')) & (gnomad4["mane_select"] == True)]
gnomad4["Entrez"] = gnomad4["gene"].map(GeneSymbol2Entrez).fillna(0).astype(int)
gnomad4 = gnomad4[gnomad4["Entrez"] != 0]
gnomad4 = gnomad4[["Entrez", "gene", "lof.pLI", "lof.z_score", "lof.oe_ci.upper"]].copy()

gnomad2 = pd.read_csv("/home/jw3514/Work/data/gnomad/gnomad.v2.1.1.lof_metrics.by_gene.txt", sep="\t")
gnomad2["Entrez"] = gnomad2["gene"].map(GeneSymbol2Entrez).fillna(0).astype(int)
gnomad2 = gnomad2[["Entrez", "gene", "pLI", "lof_z", "oe_lof_upper"]].copy()
gnomad2.columns = ["Entrez", "gene", "lof.pLI", "lof.z_score", "lof.oe_ci.upper"]

# Merge: gnomad4 primary, fill missing from gnomad2
gnomad = gnomad4.copy()
missing_in_v4 = set(gnomad2["Entrez"]) - set(gnomad4["Entrez"])
gnomad = pd.concat([gnomad, gnomad2[gnomad2["Entrez"].isin(missing_in_v4)]], ignore_index=True)
gnomad = gnomad.drop_duplicates(subset="Entrez", keep="first")
gnomad = gnomad.sort_values("lof.z_score", ascending=False)

# Check coverage
ASD_missing = set(ASD_GW.keys()) - set(gnomad["Entrez"])
SCZ_missing = set(SCZ_GW.keys()) - set(gnomad["Entrez"])
print(f"ASD genes missing from gnomAD: {len(ASD_missing)}")
print(f"SCZ genes missing from gnomAD: {len(SCZ_missing)}")

In [ ]:
# Load BrainSpan expression
BrainSpan = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat2/ExpMatch/BrainSpan.MatchDF.csv", index_col=0)

# Build annotated gene DataFrames
def annotate_gene_df(gw_dict, gnomad_df, brainspan_df):
    """Create annotated DataFrame for a gene weight set."""
    df = gnomad_df[gnomad_df["Entrez"].isin(gw_dict.keys())].copy()
    df["GW"] = df["Entrez"].map(gw_dict)
    df["BrainSpan"] = df["Entrez"].map(
        lambda x: brainspan_df.loc[x, "WB"] if x in brainspan_df.index else np.nan
    )
    return df

ASD_Genes = annotate_gene_df(ASD_GW, gnomad, BrainSpan)
SCZ_Genes = annotate_gene_df(SCZ_GW, gnomad, BrainSpan)
print(f"ASD annotated genes: {len(ASD_Genes)}, SCZ annotated genes: {len(SCZ_Genes)}")

## 3. Fast Vectorized Bias Correlation

Pre-compute numpy arrays for the expression matrix and neuronal cell type mask.
This avoids repeated DataFrame operations (AnnotateCTDat, GetSingeCellBiasCorr)
in the inner loop, giving ~100x speedup.

In [ ]:
# Pre-compute numpy structures for fast bias correlation
expr_np = HCT_Z2_MAT.values  # (n_genes, n_cell_types)
expr_gene_list = list(HCT_Z2_MAT.index)
expr_gene_set = set(expr_gene_list)
expr_gene_to_row = {g: i for i, g in enumerate(expr_gene_list)}

ct_cols = HCT_Z2_MAT.columns.values
neur_col_mask = np.array([int(c) in Neur_idx for c in ct_cols])
print(f"Neuronal cell types: {neur_col_mask.sum()} / {len(ct_cols)}")


def fast_bias_corr(asd_entrez, asd_weights, scz_entrez, scz_weights,
                   expr_np, expr_gene_to_row, neur_col_mask):
    """Compute ASD-SCZ neuronal bias Spearmans' R using vectorized numpy."""
    # ASD bias vector
    asd_rows = np.array([expr_gene_to_row[g] for g in asd_entrez])
    asd_bias = np.average(expr_np[asd_rows], axis=0, weights=asd_weights)

    # SCZ bias vector
    scz_rows = np.array([expr_gene_to_row[g] for g in scz_entrez])
    scz_bias = np.average(expr_np[scz_rows], axis=0, weights=scz_weights)

    # Spearman on neuronal cell types only
    r, _ = stats.spearmanr(asd_bias[neur_col_mask], scz_bias[neur_col_mask])
    return r


def prepare_gene_arrays(genes_df):
    """Convert gene DataFrame to numpy arrays (entrez IDs, weights), filtered to expression matrix."""
    mask = genes_df["Entrez"].isin(expr_gene_set)
    entrez = genes_df.loc[mask, "Entrez"].values
    weights = genes_df.loc[mask, "GW"].values
    return entrez, weights


def prepare_all_gene_arrays(gw_dict):
    """Convert full gene weight dict to numpy arrays, filtered to expression matrix."""
    entrez = np.array([g for g in gw_dict if g in expr_gene_set])
    weights = np.array([gw_dict[g] for g in entrez])
    return entrez, weights


# Pre-compute full gene arrays (all genes, not just those in gnomAD)
asd_all_entrez, asd_all_weights = prepare_all_gene_arrays(ASD_GW)
scz_all_entrez, scz_all_weights = prepare_all_gene_arrays(SCZ_GW)
print(f"Full gene sets: ASD={len(asd_all_entrez)}, SCZ={len(scz_all_entrez)}")


def compute_removal_curve(asd_sorted_df, scz_sorted_df, n_steps=31):
    """Compute ASD-SCZ correlation as genes are progressively removed from the top.

    Genes without annotation (not in sorted_df) are always retained in the bias
    calculation so the step-0 correlation matches the full-dataset baseline.
    """
    # Ranked genes that can be removed
    asd_ranked_entrez, asd_ranked_weights = prepare_gene_arrays(asd_sorted_df)
    scz_ranked_entrez, scz_ranked_weights = prepare_gene_arrays(scz_sorted_df)

    # Unranked genes (in full set but not in ranked df) — always kept
    asd_ranked_set = set(asd_ranked_entrez)
    scz_ranked_set = set(scz_ranked_entrez)
    asd_kept_mask = np.array([g not in asd_ranked_set for g in asd_all_entrez])
    scz_kept_mask = np.array([g not in scz_ranked_set for g in scz_all_entrez])
    asd_kept_entrez, asd_kept_weights = asd_all_entrez[asd_kept_mask], asd_all_weights[asd_kept_mask]
    scz_kept_entrez, scz_kept_weights = scz_all_entrez[scz_kept_mask], scz_all_weights[scz_kept_mask]

    correlations = []
    for i in range(n_steps):
        # Combine: unranked (always kept) + ranked genes after removal
        asd_e = np.concatenate([asd_kept_entrez, asd_ranked_entrez[i:]])
        asd_w = np.concatenate([asd_kept_weights, asd_ranked_weights[i:]])
        scz_e = np.concatenate([scz_kept_entrez, scz_ranked_entrez[i:]])
        scz_w = np.concatenate([scz_kept_weights, scz_ranked_weights[i:]])

        r = fast_bias_corr(asd_e, asd_w, scz_e, scz_w,
                           expr_np, expr_gene_to_row, neur_col_mask)
        correlations.append(r)
    return correlations

## 4. Random Removal Null Distribution

For each of 1000 permutations, randomly shuffle gene order for both ASD and SCZ,
progressively remove 0-30 genes, and recompute the bias correlation.

In [ ]:
N_REMOVAL_STEPS = 31
N_PERMS = 1000
N_PROCESSES = 10
SMOOTH = False  # Set True to apply moving average smoothing to removal curves

CACHE_FILE = PROJ_DIR / "dat/Other/ASD_SCZ_RandomGeneRemoval_Null.npy"


def _run_random_removal_batch(args):
    """Run random gene removal for a batch of permutation indices (worker function).

    Uses the FULL gene set for bias calculation. At each step, removes i genes
    (randomly ordered) from the full set.
    """
    batch_indices, asd_entrez, asd_weights, scz_entrez, scz_weights, \
        expr_np_local, gene_to_row, neur_mask, n_steps = args

    n_asd = len(asd_entrez)
    n_scz = len(scz_entrez)
    results = np.empty((len(batch_indices), n_steps))

    for bi, idx in enumerate(batch_indices):
        rng = np.random.RandomState(idx)
        asd_perm = rng.permutation(n_asd)
        scz_perm = rng.permutation(n_scz)

        asd_e_shuf = asd_entrez[asd_perm]
        asd_w_shuf = asd_weights[asd_perm]
        scz_e_shuf = scz_entrez[scz_perm]
        scz_w_shuf = scz_weights[scz_perm]

        for i in range(n_steps):
            asd_rows = np.array([gene_to_row[g] for g in asd_e_shuf[i:]])
            asd_bias = np.average(expr_np_local[asd_rows], axis=0, weights=asd_w_shuf[i:])
            scz_rows = np.array([gene_to_row[g] for g in scz_e_shuf[i:]])
            scz_bias = np.average(expr_np_local[scz_rows], axis=0, weights=scz_w_shuf[i:])
            r, _ = stats.spearmanr(asd_bias[neur_mask], scz_bias[neur_mask])
            results[bi, i] = r

    return results


# Force recompute since gene set changed
CACHE_FILE = PROJ_DIR / "dat/Other/ASD_SCZ_RandomGeneRemoval_Null_v2.npy"

if CACHE_FILE.exists():
    print(f"Loading cached null from {CACHE_FILE}")
    RandNull = np.load(CACHE_FILE)
    print(f"Null shape: {RandNull.shape} ({RandNull.shape[0]} perms x {RandNull.shape[1]} removal steps)")
else:
    print(f"Computing random removal null ({N_PERMS} perms x {N_REMOVAL_STEPS} steps, {N_PROCESSES} processes)...")

    # Use ALL genes (not just gnomAD-annotated) for the random null
    batch_size = N_PERMS // N_PROCESSES
    batches = [np.arange(i, min(i + batch_size, N_PERMS)) for i in range(0, N_PERMS, batch_size)]

    # Package args for each worker — using full gene arrays
    worker_args = [
        (batch, asd_all_entrez, asd_all_weights, scz_all_entrez, scz_all_weights,
         expr_np, expr_gene_to_row, neur_col_mask, N_REMOVAL_STEPS)
        for batch in batches
    ]

    with Pool(processes=N_PROCESSES) as pool:
        results = pool.map(_run_random_removal_batch, worker_args)

    RandNull = np.vstack(results)

    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(CACHE_FILE, RandNull)
    print(f"Saved null to {CACHE_FILE}, shape: {RandNull.shape}")

rand_mean = RandNull.mean(axis=0)
rand_std = RandNull.std(axis=0)

## 5. Gene Removal by LOEUF (Constraint) — Main Result

LOEUF (Loss-of-function Observed/Expected Upper bound Fraction) measures gene constraint.
Lower LOEUF = more constrained = less tolerant of LoF mutations.

In [ ]:
# Sort genes by LOEUF
ASD_by_LOEUF = ASD_Genes.sort_values("lof.oe_ci.upper", ascending=True)  # most constrained first
SCZ_by_LOEUF = SCZ_Genes.sort_values("lof.oe_ci.upper", ascending=True)

# Remove most constrained first (ascending LOEUF = most constrained at top)
Y_LOEUF_constrained_first = compute_removal_curve(ASD_by_LOEUF, SCZ_by_LOEUF)

# Remove least constrained first (reverse order)
ASD_by_LOEUF_rev = ASD_Genes.sort_values("lof.oe_ci.upper", ascending=False)
SCZ_by_LOEUF_rev = SCZ_Genes.sort_values("lof.oe_ci.upper", ascending=False)
Y_LOEUF_unconstrained_first = compute_removal_curve(ASD_by_LOEUF_rev, SCZ_by_LOEUF_rev)

X = list(range(N_REMOVAL_STEPS))

In [ ]:
# Smoothing helper
def moving_average(data, window_size=5):
    """Smooth 1D array with moving average (odd window)."""
    kernel = np.ones(window_size) / window_size
    padded = np.pad(data, (window_size // 2,), mode='edge')
    return np.convolve(padded, kernel, mode='valid')

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(9.5, 6), facecolor='none')
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)

Y_constrained_plot = moving_average(Y_LOEUF_constrained_first, window_size=5) if SMOOTH else Y_LOEUF_constrained_first
Y_unconstrained_plot = moving_average(Y_LOEUF_unconstrained_first, window_size=5) if SMOOTH else Y_LOEUF_unconstrained_first

ax.plot(X, Y_constrained_plot, label="Remove most constrained genes first",
        color="red", linestyle='-', marker='o', markersize=8,
        markeredgecolor='black', markeredgewidth=1, zorder=10)
ax.plot(X, Y_unconstrained_plot, label="Remove least constrained genes first",
        color="blue", linestyle='--', marker='s', markersize=8,
        markeredgecolor='black', markeredgewidth=1, zorder=10)
ax.errorbar(X, rand_mean, yerr=rand_std, fmt='-', color="grey", ecolor='grey',
            elinewidth=2, capsize=4, capthick=2, label="Random removal", zorder=5)

ax.set_xlabel("Number of Genes Removed", fontsize=25)
ax.set_ylabel("Mutation Bias Correlation", fontsize=25)
ax.legend(fontsize=18, loc='best', frameon=False)
ax.tick_params(axis='both', labelsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1.0)
ax.spines['bottom'].set_color('black')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(str(PROJ_DIR / "results/figures/LOEUF_gene_removal_ASD_SCZ_correlation.png"),
            dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# Permutation p-values at key removal points
for idx in [15, 20, 25]:
    z_c, p_c, _ = GetPermutationP(RandNull[:, idx], Y_LOEUF_constrained_first[idx], greater_than=False)
    z_u, p_u, _ = GetPermutationP(RandNull[:, idx], Y_LOEUF_unconstrained_first[idx], greater_than=True)
    print(f"After removing {idx} genes:")
    print(f"  Constrained first: r={Y_LOEUF_constrained_first[idx]:.3f}, z={z_c:.2f}, p={p_c:.4f}")
    print(f"  Unconstrained first: r={Y_LOEUF_unconstrained_first[idx]:.3f}, z={z_u:.2f}, p={p_u:.4f}")

## 6. Gene Removal by BrainSpan Expression — Supporting Result

In [ ]:
ASD_by_BS = ASD_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=False)  # highest first
SCZ_by_BS = SCZ_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=False)

# Remove highest expressed first
Y_BS_high_first = compute_removal_curve(ASD_by_BS, SCZ_by_BS)

# Remove lowest expressed first
ASD_by_BS_rev = ASD_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=True)
SCZ_by_BS_rev = SCZ_Genes.dropna(subset=["BrainSpan"]).sort_values("BrainSpan", ascending=True)
Y_BS_low_first = compute_removal_curve(ASD_by_BS_rev, SCZ_by_BS_rev)

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(9.5, 6), facecolor='none')
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)

Y_low_plot = moving_average(Y_BS_low_first, window_size=5) if SMOOTH else Y_BS_low_first
Y_high_plot = moving_average(Y_BS_high_first, window_size=5) if SMOOTH else Y_BS_high_first

ax.plot(X, Y_low_plot, label="Remove lowest expressed genes first",
        color="red", linestyle='-', marker='o', markersize=8,
        markeredgecolor='black', markeredgewidth=1, zorder=10)
ax.plot(X, Y_high_plot, label="Remove highest expressed genes first",
        color="blue", linestyle='--', marker='s', markersize=8,
        markeredgecolor='black', markeredgewidth=1, zorder=10)
ax.errorbar(X, rand_mean, yerr=rand_std, fmt='-', color="grey", ecolor='grey',
            elinewidth=2, capsize=4, capthick=2, label="Random removal", zorder=5)

ax.set_xlabel("Number of Genes Removed", fontsize=25)
ax.set_ylabel("Mutation Bias Correlation", fontsize=25)
ax.legend(fontsize=18, loc='best', frameon=False)
ax.tick_params(axis='both', labelsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1.0)
ax.spines['bottom'].set_color('black')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(str(PROJ_DIR / "results/figures/BrainSpan_gene_removal_ASD_SCZ_correlation.png"),
            dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
for idx in [15, 20, 25]:
    z_low, p_low, _ = GetPermutationP(RandNull[:, idx], Y_BS_low_first[idx], greater_than=True)
    z_high, p_high, _ = GetPermutationP(RandNull[:, idx], Y_BS_high_first[idx], greater_than=False)
    print(f"After removing {idx} genes:")
    print(f"  Lowest expressed first: r={Y_BS_low_first[idx]:.3f}, z={z_low:.2f}, p={p_low:.4f}")
    print(f"  Highest expressed first: r={Y_BS_high_first[idx]:.3f}, z={z_high:.2f}, p={p_high:.4f}")

## 7. Constrained vs Unconstrained Gene Subsets

In [ ]:
LOEUF_THRESHOLD = 0.2

ASD_Constrained = ASD_Genes[ASD_Genes["lof.oe_ci.upper"] < LOEUF_THRESHOLD]
ASD_Unconstrained = ASD_Genes[ASD_Genes["lof.oe_ci.upper"] >= LOEUF_THRESHOLD]
SCZ_Constrained = SCZ_Genes[SCZ_Genes["lof.oe_ci.upper"] < LOEUF_THRESHOLD]
SCZ_Unconstrained = SCZ_Genes[SCZ_Genes["lof.oe_ci.upper"] >= LOEUF_THRESHOLD]

print(f"ASD: {len(ASD_Constrained)} constrained, {len(ASD_Unconstrained)} unconstrained (threshold={LOEUF_THRESHOLD})")
print(f"SCZ: {len(SCZ_Constrained)} constrained, {len(SCZ_Unconstrained)} unconstrained")

# Constrained genes only
gw_asd_c = dict(zip(ASD_Constrained["Entrez"], ASD_Constrained["GW"]))
gw_scz_c = dict(zip(SCZ_Constrained["Entrez"], SCZ_Constrained["GW"]))
bias_asd_c = AnnotateCTDat(HumanCT_AvgZ_Weighted(HCT_Z2_MAT, gw_asd_c), Anno)
bias_scz_c = AnnotateCTDat(HumanCT_AvgZ_Weighted(HCT_Z2_MAT, gw_scz_c), Anno)
r_c, p_c = GetSingeCellBiasCorr(bias_asd_c, bias_scz_c, efflabel="EFFECT", CTs=Neur_idx)
print(f"\nConstrained genes (LOEUF < {LOEUF_THRESHOLD}): r={r_c:.3f}, p={p_c:.2e}")

# Unconstrained genes only
gw_asd_u = dict(zip(ASD_Unconstrained["Entrez"], ASD_Unconstrained["GW"]))
gw_scz_u = dict(zip(SCZ_Unconstrained["Entrez"], SCZ_Unconstrained["GW"]))
bias_asd_u = AnnotateCTDat(HumanCT_AvgZ_Weighted(HCT_Z2_MAT, gw_asd_u), Anno)
bias_scz_u = AnnotateCTDat(HumanCT_AvgZ_Weighted(HCT_Z2_MAT, gw_scz_u), Anno)
r_u, p_u = GetSingeCellBiasCorr(bias_asd_u, bias_scz_u, efflabel="EFFECT", CTs=Neur_idx)
print(f"Unconstrained genes (LOEUF >= {LOEUF_THRESHOLD}): r={r_u:.3f}, p={p_u:.2e}")
print(f"\nBaseline (all genes): r={r_baseline:.3f}")